# 01 - Dataset: legendas com BLIP

Gera legendas rascunho com BLIP, exporto pra revisar e depois monto o legendas.txt final.

## 1. Clonar o repo

In [ ]:
!git clone https://github.com/lramosc1512/atelie-generativo-LeonidIA.git
%cd atelie-generativo-LeonidIA
!ls dados/imagens | head
!wc -l dados/fontes.csv

## 2. Instalar libs

In [ ]:
!pip -q install transformers pillow pandas

## 2.5 Login HF

In [ ]:
from huggingface_hub import login
login()

## 3. Carregar o BLIP

In [ ]:
import torch
import pandas as pd
from pathlib import Path
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando device: {device}")

proc = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
).to(device)

## 4. Gerar legendas rascunho

In [ ]:
IMG_DIR = Path("dados/imagens")
extensoes = {".jpg", ".jpeg", ".png"}
arquivos = sorted([p for p in IMG_DIR.iterdir() if p.suffix.lower() in extensoes])
print(f"{len(arquivos)} imagens encontradas.")

rascunhos = []
for caminho in arquivos:
    img = Image.open(caminho).convert("RGB")
    entradas = proc(img, return_tensors="pt").to(device)
    saida = blip.generate(**entradas, max_new_tokens=40)
    legenda_bruta = proc.decode(saida[0], skip_special_tokens=True)
    rascunhos.append({"arquivo": caminho.name, "legenda_blip_en": legenda_bruta, "legenda_revisada": ""})
    print(f"{caminho.name}: {legenda_bruta}")

df_rascunho = pd.DataFrame(rascunhos)
df_rascunho.to_csv("dados/legendas_rascunho.csv", index=False)
df_rascunho.head()

## 5. Baixar rascunho pra revisar

Revisar em português, começando cada legenda com `estilo_brasilia,`.

In [ ]:
from google.colab import files
files.download("dados/legendas_rascunho.csv")

## 6. Subir o CSV revisado

In [ ]:
from google.colab import files
enviado = files.upload()  # selecione o CSV revisado
nome_arquivo_revisado = list(enviado.keys())[0]
df_revisado = pd.read_csv(nome_arquivo_revisado)
df_revisado.head()

## 7. Validar

Confere se falta token, legenda vazia ou arquivo sem entrada no fontes.csv.

In [ ]:
TOKEN = "estilo_brasilia,"
problemas = []

# 1) legendas vazias
vazias = df_revisado[df_revisado["legenda_revisada"].isna() | (df_revisado["legenda_revisada"].str.strip() == "")]
for _, linha in vazias.iterrows():
    problemas.append(f'Legenda vazia: {linha["arquivo"]}')

# 2) token no inicio
com_legenda = df_revisado.dropna(subset=["legenda_revisada"])
sem_token = com_legenda[~com_legenda["legenda_revisada"].str.strip().str.startswith(TOKEN)]
for _, linha in sem_token.iterrows():
    problemas.append(f'Falta o token "{TOKEN}" no inicio: {linha["arquivo"]}')

# 3) arquivo existe em fontes.csv
fontes = pd.read_csv("dados/fontes.csv")
nomes_fontes = set(fontes["nome_arquivo"])
nomes_legendas = set(df_revisado["arquivo"])
faltando_em_fontes = nomes_legendas - nomes_fontes
for nome in faltando_em_fontes:
    problemas.append(f'Arquivo sem entrada correspondente em fontes.csv: {nome}')

if problemas:
    print(f"{len(problemas)} problema(s) encontrado(s):\n")
    for p in problemas:
        print(f"  - {p}")
else:
    print("Tudo certo! Pode seguir para o Passo 8.")

## 8. Gerar legendas.txt final

In [ ]:
assert not problemas, "Corrija os problemas do Passo 7 antes de continuar."

with open("dados/legendas.txt", "w", encoding="utf-8") as f:
    for _, linha in df_revisado.iterrows():
        f.write(f'{linha["arquivo"]},{linha["legenda_revisada"].strip()}\n')

print("dados/legendas.txt gerado com sucesso.")
!head dados/legendas.txt

## 9. Commitar

```bash
git add dados/legendas.txt
git commit -m "legendas revisadas"
git push
```

In [ ]:
from google.colab import files
files.download("dados/legendas.txt")